In [ ]:
rows = []
for cond, mice in zip(('ctrl', 'ko'),(ctrl_mice,ko_mice)):
    for mouse in mice:
        print(mouse)
        for day in range(6):
            sess = u.load_single_day(mouse, day, trial_mat_keys=['F_dff',], timeseries_keys=['F_dff',], verbose = False)
            
            rz_early = (np.argwhere(sess.trial_matrices['bin_edges'][:-1]>=sess.rzone_early['tfront'])[0], np.argwhere(sess.rzone_early['tback']<=sess.trial_matrices['bin_edges'][1:])[0] )
            rz_late = (np.argwhere(sess.trial_matrices['bin_edges'][:-1]>=sess.rzone_late['tfront'])[0], 29 )

            if sess.novel_arm == 1: 
                rz_nov = rz_late
                rz_fam = rz_early
            else: 
                rz_nov = rz_early
                rz_fam = rz_late

            for ttype in ('nov', 'fam'):
                if ttype == 'nov':
                    rz = rz_nov
                    coef = 1
                    pc_mask = sess.nov_place_cell_mask()
                else:
                    rz = rz_fam
                    coef = -1 
                    pc_mask = sess.fam_place_cell_mask()

                tmat = sess.trial_matrices['F_dff'][sess.trial_info['LR']==coef*sess.novel_arm, :, :]
                tmat[np.isnan(tmat)]=0

                cell_corrs = []
                for cell in range(tmat.shape[-1]):
                    _mat = np.corrcoef(tmat[:,:,cell])
                    _mat[np.diag_indices_from(_mat)]=np.nan
                    cell_corrs.append(np.nanmean(_mat.ravel()))
                cell_corrs = np.array(cell_corrs)

                cell_pos = np.nanargmax(np.nanmean(sess.trial_matrices['F_dff'][sess.trial_info['LR']==(coef*sess.novel_arm), :, :], axis=0),axis=0)
                
                r_mask = (cell_pos>=(rz[0]-5)) * (cell_pos<=(rz[0]+1))

                
                rows.append({'condition': cond,
                            'mouse': mouse,
                            'day': day,
                            'ttype': ttype,
                            'rzone_corr': np.nanmean(cell_corrs[r_mask*pc_mask]),
                            'nonrzone_corr': np.nanmean(cell_corrs[~r_mask]),
                            'nonrzone_pc_corr': np.nanmean(cell_corrs[(~r_mask)*pc_mask])})
                
df = pd.DataFrame(rows)

In [ ]:

fig, ax = plt.subplots(1,2,figsize=[9,3], sharex=True, sharey=True)
_ = sns.stripplot(data = df.loc[df['ttype']=='fam'],
                  x='day',
                  y='rzone_corr',
                  hue = 'condition',
                  hue_order= ['ctrl','ko'],
                  palette=['black','red'],
                  ax=ax[0], dodge=True, alpha=.4, s=5)

tmp_df_mu = df.groupby(['condition', 'day', 'ttype'])['rzone_corr'].mean().reset_index()
sns.pointplot(data=tmp_df_mu.loc[tmp_df_mu['ttype']=='fam'],
            x='day',
            y='rzone_corr',
            errorbar=None,
            hue='condition',
            hue_order = ['ctrl', 'ko'],
            dodge = .4,
            palette = ['black', 'red'],
            ax=ax[0],
            linestyles="none",
            marker="_", markersize=15, markeredgewidth=5, legend=False, alpha=1.,
            )


_ = sns.stripplot(data = df.loc[df['ttype']=='nov'],
                  x='day',
                  y='rzone_corr',
                  hue = 'condition',
                  hue_order= ['ctrl','ko'],
                  palette=['black','red'],
                  ax=ax[1], dodge=True, alpha=.4, s=8, marker='P')

sns.pointplot(data=tmp_df_mu.loc[tmp_df_mu['ttype']=='nov'],
            x='day',
            y='rzone_corr',
            errorbar=None,
            hue='condition',
            hue_order = ['ctrl', 'ko'],
            dodge = .4,
            palette = ['black', 'red'],
            ax=ax[1],
            linestyles="none",
            marker="_", markersize=15, markeredgewidth=5, legend=False, alpha=1.,
            )

fig, ax = plt.subplots(1,2,figsize=[9,3], sharex=True, sharey=True)
_ = sns.stripplot(data = df.loc[df['ttype']=='fam'],
                  x='day',
                  y='nonrzone_pc_corr',
                  hue = 'condition',
                  hue_order= ['ctrl','ko'],
                  palette=['black','red'],
                  ax=ax[0], dodge=True, alpha=.4, s=5)

tmp_df_mu = df.groupby(['condition', 'day', 'ttype'])['nonrzone_pc_corr'].mean().reset_index()
sns.pointplot(data=tmp_df_mu.loc[tmp_df_mu['ttype']=='fam'],
            x='day',
            y='nonrzone_pc_corr',
            errorbar=None,
            hue='condition',
            hue_order = ['ctrl', 'ko'],
            dodge = .4,
            palette = ['black', 'red'],
            ax=ax[0],
            linestyles="none",
            marker="_", markersize=15, markeredgewidth=5, legend=False, alpha=1.,
            )


_ = sns.stripplot(data = df.loc[df['ttype']=='nov'],
                  x='day',
                  y='nonrzone_pc_corr',
                  hue = 'condition',
                  hue_order= ['ctrl','ko'],
                  palette=['black','red'],
                  ax=ax[1], dodge=True, alpha=.4, s=8, marker='P')

sns.pointplot(data=tmp_df_mu.loc[tmp_df_mu['ttype']=='nov'],
            x='day',
            y='nonrzone_pc_corr',
            errorbar=None,
            hue='condition',
            hue_order = ['ctrl', 'ko'],
            dodge = .4,
            palette = ['black', 'red'],
            ax=ax[1],
            linestyles="none",
            marker="_", markersize=15, markeredgewidth=5, legend=False, alpha=1.,
            )

In [ ]:
# aov = pg.mixed_anova(data=df.loc[(df['day']<6) * (df['ttype']=='nov')],
#                       dv='rzone_corr', subject='mouse', within='day', between='condition')
# print(aov)

res = pg.pairwise_tests(data=df.loc[(df['day']<6)], # * (df['ttype']=='nov')],
                        dv='rzone_corr', subject='mouse', within='day', between='condition',
                        parametric=False, padjust='holm')
res

In [ ]:
res = pg.pairwise_tests(data=df.loc[(df['day']<6)], # * (df['ttype']=='nov')],
                        dv='nonrzone_pc_corr', subject='mouse', within='day', between='condition',
                        parametric=False, padjust='holm')
res

In [ ]:
fig, ax = plt.subplots(1,2,figsize=[9,3], sharex=True, sharey=True)
_ = sns.stripplot(data = df.loc[df['ttype']=='fam'],
                  x='day',
                  y='nonrzone_pc_corr',
                  hue = 'condition',
                  hue_order= ['ctrl','ko'],
                  palette=['black','red'],
                  ax=ax[0], dodge=True, alpha=.4, s=5)

tmp_df_mu = df.groupby(['condition', 'day', 'ttype'])['nonrzone_pc_corr'].mean().reset_index()
sns.pointplot(data=tmp_df_mu.loc[tmp_df_mu['ttype']=='fam'],
            x='day',
            y='nonrzone_pc_corr',
            errorbar=None,
            hue='condition',
            hue_order = ['ctrl', 'ko'],
            dodge = .4,
            palette = ['black', 'red'],
            ax=ax[0],
            linestyles="none",
            marker="_", markersize=15, markeredgewidth=5, legend=False, alpha=1.,
            )


_ = sns.stripplot(data = df.loc[df['ttype']=='nov'],
                  x='day',
                  y='nonrzone_pc_corr',
                  hue = 'condition',
                  hue_order= ['ctrl','ko'],
                  palette=['black','red'],
                  ax=ax[1], dodge=True, alpha=.4, s=8, marker='P')

sns.pointplot(data=tmp_df_mu.loc[tmp_df_mu['ttype']=='nov'],
            x='day',
            y='nonrzone_pc_corr',
            errorbar=None,
            hue='condition',
            hue_order = ['ctrl', 'ko'],
            dodge = .4,
            palette = ['black', 'red'],
            ax=ax[1],
            linestyles="none",
            marker="_", markersize=15, markeredgewidth=5, legend=False, alpha=1.,
            )

In [ ]:
rows = []

for mouse in sparse_mice:
    print(mouse)
    for day in range(6):
        if mouse == 'SparseKO_09' and day ==2:
            continue
        for chan in ('channel_0','channel_1'):
            sess = u.load_single_day(mouse, day, trial_mat_keys=[f'{chan}_F_dff',], timeseries_keys=[f'{chan}_F_dff',], verbose = False)
            
            rz_early = (np.argwhere(sess.trial_matrices['bin_edges'][:-1]>=sess.rzone_early['tfront'])[0], np.argwhere(sess.rzone_early['tback']<=sess.trial_matrices['bin_edges'][1:])[0] )
            rz_late = (np.argwhere(sess.trial_matrices['bin_edges'][:-1]>=sess.rzone_late['tfront'])[0], 29 )

            if sess.novel_arm == 1: 
                rz_nov = rz_late
                rz_fam = rz_early
            else: 
                rz_nov = rz_early
                rz_fam = rz_late

            pc_mask = sess.nov_place_cell_mask(mux=True, chan=chan) + sess.fam_place_cell_mask(mux=True, chan=chan)
            # place cell peak

            tmat_nov = sess.trial_matrices[f'{chan}_F_dff'][sess.trial_info["LR"]==sess.novel_arm,:,:][:,:,pc_mask]
            maxpos_nov = np.nanargmax(np.nanmean(tmat_nov,axis=0), axis=0)
            rz_cell_nov = (maxpos_nov>=(rz_nov[0]-6)) * (maxpos_nov<=(rz_nov[0]+2))

            tmat_fam = sess.trial_matrices[f'{chan}_F_dff'][sess.trial_info["LR"]!=sess.novel_arm,:,:][:,:,pc_mask]
            maxpos_fam = np.nanargmax(np.nanmean(tmat_fam,axis=0), axis=0)
            rz_cell_fam = (maxpos_nov>=(rz_fam[0]-6)) * (maxpos_nov<=(rz_fam[0]+2))

            r_cell_mask = rz_cell_fam * rz_cell_nov

            cell_corrs = {'nov': [], 'fam': []}
            for cell in range(tmat_nov.shape[-1]):
                _mat = np.corrcoef(tmat_nov[:,:,cell])
                _mat[np.diag_indices_from(_mat)]=np.nan
                cell_corrs['nov'].append(np.nanmean(_mat.ravel()))

                _mat = np.corrcoef(tmat_fam[:,:,cell])
                _mat[np.diag_indices_from(_mat)]=np.nan
                cell_corrs['fam'].append(np.nanmean(_mat.ravel()))
            cell_corrs = {k:np.array(v) for k,v in cell_corrs.items()}

            for ttype in ('nov', 'fam'):
                # if ttype == 'nov':
                #     cell_mask = rz_cell_nov
                # else:
                #     cell_mask = rz_cell_fam
                rows.append({'chan': chan,
                            'mouse': mouse,
                            'day': day,
                            'ttype': ttype,
                            'rzone_corr': np.nanmean(cell_corrs[ttype][r_cell_mask]),
                            'nonrzone_corr': np.nanmean(cell_corrs[ttype][~r_cell_mask]),
                            'all_corr': np.nanmean(cell_corrs[ttype]),
                            })
                
df = pd.DataFrame(rows)

In [ ]:
fig, ax = plt.subplots(1,2,figsize=[9,3], sharex=True, sharey=True)
_ = sns.stripplot(data = df.loc[df['ttype']=='fam'],
                  x='day',
                  y='all_corr',
                  hue = 'chan',
                  hue_order= ['channel_1','channel_0'],
                  palette=['black','red'],
                  ax=ax[0], dodge=True, alpha=.4, s=5)

for d in range(0,12,2):
    for (x0, y0), (x1, y1) in zip(ax[0].collections[d].get_offsets(), ax[0].collections[d+1].get_offsets()):
        ax[0].plot([x0, x1], [y0, y1], color='black', ls=':', zorder=0)

_ = sns.stripplot(data = df.loc[df['ttype']=='nov'],
                  x='day',
                  y='all_corr',
                  hue = 'chan',
                  hue_order= ['channel_1','channel_0'],
                  palette=['black','red'],
                  ax=ax[1], dodge=True, alpha=.4, s=8, marker='P')
for d in range(0,12,2):
    for (x0, y0), (x1, y1) in zip(ax[1].collections[d].get_offsets(), ax[1].collections[d+1].get_offsets()):
        ax[1].plot([x0, x1], [y0, y1], color='black', ls=':', zorder=0)

